# Phần 2: chạy BM25 trên pool Test và Train (Kaggle CPU)

**Notebook này là bản ghi lại lần chạy thật trên Kaggle, dùng để xem lại. Không có ý định chạy lại trên máy cá nhân** (cần dataset pool và code đã tải lên Kaggle).

Đây là **Phần 2** của Pipeline 1, nằm sau Phần 1 (dựng pool, làm ở notebook Colab). Kết quả của notebook này (danh sách top-100 tài liệu cho từng câu hỏi) được dùng ở phần train để khai thác cặp âm khó.


**Việc notebook này làm:** đo BM25 tìm đúng tài liệu nguồn tốt đến đâu trên pool Test (300 parent, 2.606 câu hỏi) và pool Train (500 parent, 4.322 câu hỏi), với 2 cách cắt đoạn:
- `fixed_window_bm25`: cắt cửa sổ cố định 256 từ, bước 128.
- `structure_bm25`: cắt theo cấu trúc văn bản (tối đa 1.500 ký tự mỗi đoạn).

Sau đó tách kết quả theo loại đạo văn (artificial / translation / simulated) và dùng bootstrap cặp để kiểm tra chênh lệch giữa 2 cách cắt có đáng tin không.

**Dữ liệu dùng lại:** pool Test và Train đã dựng và kiểm chứng sạch ở Colab (0 tài liệu trùng với các tập DEV). Notebook này chỉ đọc chúng, không dựng lại.

**Các bước:**
1. Copy code, vá lỗi `read_jsonl`.
2. Thay `bm25.py` bằng bản nhanh.
3. Copy pool, duyệt governance.
4. Chạy BM25 (Test rồi Train).
5. Đọc kết quả tổng, nén kết quả.
6. Lấy nhãn loại đạo văn, tách kết quả theo loại.
7. Bootstrap cặp: fixed-window có thật sự hơn structure không.

### Bước 1 — Copy code sang vùng ghi được, vá lỗi `read_jsonl`, cài thư viện

`/kaggle/input/...` là vùng CHỈ ĐỌC. Copy thư mục `research/` sang `/kaggle/working/repo` (ghi được),
rồi vá lại lỗi `read_jsonl` dùng `str.splitlines()` sai (bị cắt nhầm dòng có ký tự `\x85`) — lỗi
này đã sửa trong repo gốc, nhưng dataset code trên Kaggle là bản cũ, chưa có bản vá.

In [1]:
import glob, shutil, os

hits = glob.glob('/kaggle/input/**/requirements-research.txt', recursive=True)
assert hits, 'Không tìm thấy requirements-research.txt — kiểm tra lại đã gắn đúng dataset code chưa'
RESEARCH_SRC = os.path.dirname(hits[0])
print('Tìm thấy thư mục research/ tại:', RESEARCH_SRC)

os.makedirs('/kaggle/working/repo', exist_ok=True)
shutil.copytree(RESEARCH_SRC, '/kaggle/working/repo/research', dirs_exist_ok=True)

!pip install -q -r /kaggle/working/repo/research/requirements-research.txt

# Vá lỗi read_jsonl (giống bản đã vá trên Colab)
path = "/kaggle/working/repo/research/offline_retrieval/common.py"
src = open(path, encoding="utf-8").read()
old = 'for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):'
new = 'for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").split("\\n"), 1):'
assert src.count(old) == 1, f"tìm thấy {src.count(old)} lần, cần đúng 1"
open(path, "w", encoding="utf-8").write(src.replace(old, new))
print("Đã vá xong read_jsonl")

Tìm thấy thư mục research/ tại: /kaggle/input/datasets/yenkim1/kltn-pan-pc11-archive-raw/research_for_colab/research
Đã vá xong read_jsonl


### Bước 2 — Thay `bm25.py` bằng bản nhanh

**Vì sao phải thay:** bản BM25 cũ chạy bằng vòng lặp Python thuần, và không loại từ phổ biến ("the", "of"...) nên gần như đoạn nào cũng phải tính. Với 600.890 cửa sổ × 2.606 câu hỏi, nó chạy hơn 6 giờ vẫn chưa xong.

**Bản mới** tính bằng ma trận thưa (`scipy.sparse`): cùng công thức BM25 (k1=1,2; b=0,75), cùng cách gộp về tài liệu, cùng quy tắc xử lý hoà điểm. Đã kiểm tra trên dữ liệu ngẫu nhiên: thứ tự kết quả giống hệt bản cũ, điểm chỉ lệch cỡ 1e-12.

Bản cũ vẫn được giữ lại dưới tên `rank_bm25_reference` để đối chiếu khi cần.

*Ô dưới ghi đè file `bm25.py` trong `/kaggle/working/repo`, không đụng vào dataset gốc (chỉ đọc).*

In [2]:
%%writefile /kaggle/working/repo/research/offline_retrieval/bm25.py
"""Small deterministic Okapi BM25 baseline with no service dependency."""

from __future__ import annotations

from collections import Counter, defaultdict
import math
from typing import Any, Iterable

from .aggregation import render_ranked_candidates
from .common import tokenize


def rank_bm25_reference(
    corpus: Iterable[dict[str, Any]],
    queries: Iterable[dict[str, Any]],
    *,
    top_k: int,
    k1: float = 1.2,
    b: float = 0.75,
    group_by_document: bool = False,
) -> dict[str, list[dict[str, Any]]]:
    if top_k <= 0:
        raise ValueError("top_k must be positive")
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError("BM25 requires k1 > 0 and b in [0, 1]")

    items = list(corpus)
    if not items:
        return {str(query["query_id"]): [] for query in queries}

    document_lengths: list[int] = []
    postings: dict[str, list[tuple[int, int]]] = defaultdict(list)
    seen_ids: set[str] = set()
    for item in items:
        chunk_id = item.get("chunk_id")
        text = item.get("text")
        if not isinstance(chunk_id, str) or not chunk_id:
            raise ValueError("Every corpus item requires a non-empty chunk_id")
        if chunk_id in seen_ids:
            raise ValueError(f"Duplicate chunk_id: {chunk_id}")
        seen_ids.add(chunk_id)
        if not isinstance(text, str):
            raise ValueError(f"Corpus item {chunk_id} requires string text")
        tokens = tokenize(text)
        counts = Counter(tokens)
        document_lengths.append(len(tokens))
        for term, frequency in counts.items():
            postings[term].append((len(document_lengths) - 1, frequency))

    average_length = sum(document_lengths) / len(items)
    average_length = average_length or 1.0
    corpus_size = len(items)

    results: dict[str, list[dict[str, Any]]] = {}
    seen_query_ids: set[str] = set()
    for query in queries:
        query_id = query.get("query_id")
        query_text = query.get("text")
        if not isinstance(query_id, str) or not query_id:
            raise ValueError("Every query requires a non-empty query_id")
        if query_id in seen_query_ids:
            raise ValueError(f"Duplicate query_id: {query_id}")
        seen_query_ids.add(query_id)
        if not isinstance(query_text, str):
            raise ValueError(f"Query {query_id} requires string text")

        query_terms = Counter(tokenize(query_text))
        scores: dict[int, float] = defaultdict(float)
        for term, query_frequency in query_terms.items():
            term_postings = postings.get(term)
            if not term_postings:
                continue
            document_frequency = len(term_postings)
            inverse_document_frequency = math.log(
                1
                + (corpus_size - document_frequency + 0.5)
                / (document_frequency + 0.5)
            )
            for item_index, frequency in term_postings:
                document_length = document_lengths[item_index]
                denominator = frequency + k1 * (
                    1 - b + b * document_length / average_length
                )
                scores[item_index] += query_frequency * inverse_document_frequency * (
                    frequency * (k1 + 1) / denominator
                )

        # A non-match has no retrieval evidence. Omitting zero-score chunks also
        # avoids arbitrary chunk-id tie breaks filling a requested top-k.
        scored = [
            (score, items[item_index]["chunk_id"], items[item_index])
            for item_index, score in scores.items()
            if score > 0.0
        ]

        results[query_id] = render_ranked_candidates(
            scored,
            top_k=top_k,
            group_by_document=group_by_document,
        )
    return results


# ---------------------------------------------------------------------------
# Bản nhanh (vector hoá bằng scipy.sparse). Cùng công thức, cùng cách gộp về
# tài liệu và cùng quy tắc phá hoà như rank_bm25_reference; chỉ khác thứ tự
# cộng số thực nên điểm có thể lệch ~1e-12.
# ---------------------------------------------------------------------------
def rank_bm25(
    corpus: Iterable[dict[str, Any]],
    queries: Iterable[dict[str, Any]],
    *,
    top_k: int,
    k1: float = 1.2,
    b: float = 0.75,
    group_by_document: bool = False,
    query_batch_size: int = 32,
) -> dict[str, list[dict[str, Any]]]:
    if not group_by_document:
        return rank_bm25_reference(
            corpus, queries, top_k=top_k, k1=k1, b=b, group_by_document=False
        )
    import numpy as np
    from array import array
    from scipy import sparse

    if top_k <= 0:
        raise ValueError("top_k must be positive")
    if k1 <= 0 or not 0 <= b <= 1:
        raise ValueError("BM25 requires k1 > 0 and b in [0, 1]")
    items = list(corpus)
    queries = list(queries)
    if not items:
        return {str(query["query_id"]): [] for query in queries}

    vocab: dict[str, int] = {}
    indptr = array("q", [0])
    indices = array("i")
    term_freqs = array("d")
    lengths = array("d")
    seen_ids: set[str] = set()
    for item in items:
        chunk_id = item.get("chunk_id")
        text = item.get("text")
        if not isinstance(chunk_id, str) or not chunk_id:
            raise ValueError("Every corpus item requires a non-empty chunk_id")
        if chunk_id in seen_ids:
            raise ValueError(f"Duplicate chunk_id: {chunk_id}")
        seen_ids.add(chunk_id)
        if not isinstance(text, str):
            raise ValueError(f"Corpus item {chunk_id} requires string text")
        document_id = item.get("document_id")
        if not isinstance(document_id, str) or not document_id:
            raise ValueError("Every retrieval candidate requires document_id")
        tokens = tokenize(text)
        lengths.append(len(tokens))
        for term, frequency in Counter(tokens).items():
            term_index = vocab.get(term)
            if term_index is None:
                term_index = vocab[term] = len(vocab)
            indices.append(term_index)
            term_freqs.append(frequency)
        indptr.append(len(indices))
    del seen_ids

    n_items, n_terms = len(items), len(vocab)
    indptr_np = np.frombuffer(indptr, dtype=np.int64)
    indices_np = np.frombuffer(indices, dtype=np.int32)
    tf = np.frombuffer(term_freqs, dtype=np.float64)
    lengths_np = np.frombuffer(lengths, dtype=np.float64)
    average_length = float(lengths_np.sum()) / n_items or 1.0
    df = np.bincount(indices_np, minlength=n_terms).astype(np.float64)
    idf = np.log(1 + (n_items - df + 0.5) / (df + 0.5))
    row_lengths = np.repeat(lengths_np, np.diff(indptr_np))
    weights = idf[indices_np] * (
        tf * (k1 + 1) / (tf + k1 * (1 - b + b * row_lengths / average_length))
    )
    del row_lengths, term_freqs
    # term x item: mỗi hàng là posting list đã nhân sẵn trọng số BM25
    term_item = sparse.csr_matrix(
        (weights, indices_np, indptr_np), shape=(n_items, n_terms)
    ).T.tocsr()
    del weights

    # Gom chunk theo tài liệu; trong mỗi tài liệu xếp chunk theo chunk_id để
    # argmax lấy chunk có id nhỏ nhất khi hoà (giống bản gốc).
    doc_ids = sorted({item["document_id"] for item in items})
    doc_pos = {d: i for i, d in enumerate(doc_ids)}
    item_doc = np.fromiter(
        (doc_pos[item["document_id"]] for item in items), dtype=np.int64, count=n_items
    )
    per_doc: list[list[int]] = [[] for _ in doc_ids]
    for item_index, item in enumerate(items):
        per_doc[item_doc[item_index]].append(item_index)
    per_doc_np = [
        np.array(sorted(members, key=lambda i: items[i]["chunk_id"]), dtype=np.int64)
        for members in per_doc
    ]
    del per_doc

    results: dict[str, list[dict[str, Any]]] = {}
    seen_query_ids: set[str] = set()
    for start in range(0, len(queries), query_batch_size):
        batch = queries[start : start + query_batch_size]
        rows, cols, vals, batch_ids = [], [], [], []
        for row, query in enumerate(batch):
            query_id = query.get("query_id")
            query_text = query.get("text")
            if not isinstance(query_id, str) or not query_id:
                raise ValueError("Every query requires a non-empty query_id")
            if query_id in seen_query_ids:
                raise ValueError(f"Duplicate query_id: {query_id}")
            seen_query_ids.add(query_id)
            if not isinstance(query_text, str):
                raise ValueError(f"Query {query_id} requires string text")
            batch_ids.append(query_id)
            for term, query_frequency in Counter(tokenize(query_text)).items():
                term_index = vocab.get(term)
                if term_index is not None:
                    rows.append(row)
                    cols.append(term_index)
                    vals.append(float(query_frequency))
        query_matrix = sparse.csr_matrix(
            (vals, (rows, cols)), shape=(len(batch), n_terms)
        )
        batch_scores = (query_matrix @ term_item).tocsr()
        for row, query_id in enumerate(batch_ids):
            lo, hi = batch_scores.indptr[row], batch_scores.indptr[row + 1]
            matched = batch_scores.indices[lo:hi]
            matched_scores = batch_scores.data[lo:hi]
            keep = matched_scores > 0.0
            matched, matched_scores = matched[keep], matched_scores[keep]
            if matched.size == 0:
                results[query_id] = []
                continue
            dense = np.zeros(n_items, dtype=np.float64)
            dense[matched] = matched_scores
            best = np.full(len(doc_ids), -np.inf)
            np.maximum.at(best, item_doc[matched], matched_scores)
            candidates = np.flatnonzero(best > 0.0)
            # (-điểm, document_id): doc_ids đã sort nên chỉ số = thứ tự theo id
            order = candidates[np.lexsort((candidates, -best[candidates]))][:top_k]
            ranked = []
            for rank, doc_index in enumerate(order, 1):
                members = per_doc_np[doc_index]
                item = items[int(members[int(np.argmax(dense[members]))])]
                ranked.append(
                    {
                        "document_id": item["document_id"],
                        "supporting_chunk_id": item["chunk_id"],
                        "score": float(best[doc_index]),
                        "rank": rank,
                        "source_spans": item.get("source_spans", []),
                    }
                )
            results[query_id] = ranked
    return results

Overwriting /kaggle/working/repo/research/offline_retrieval/bm25.py


### Bước 3 — Copy pool Test/Train sang vùng ghi được, duyệt governance

- `/kaggle/input` chỉ đọc, nên phải copy 2 pool sang `/kaggle/working/data` trước.
- Code từ chối chạy nếu `dataset-manifest.json` chưa ghi `governance = APPROVED`. Đây là thiết kế cố ý với dữ liệu bên thứ ba. PAN-PC-11 là corpus công khai cho nghiên cứu, nên tác giả tự duyệt và ghi tên, ngày duyệt vào manifest để truy vết.

Kết quả đúng phải in `governance: APPROVED` cho cả 2 pool.

In [3]:
import glob, json, subprocess, pathlib, os, shutil

REPO = "/kaggle/working/repo"

def copy_pool_dir(folder_name):
    hits = glob.glob(f'/kaggle/input/**/{folder_name}', recursive=True)
    assert hits, f'Không tìm thấy {folder_name} — kiểm tra đã gắn đúng dataset mới chưa'
    src = hits[0]
    dst = f"/kaggle/working/data/{folder_name}"
    if not os.path.exists(dst):
        print("Đang copy", folder_name, "sang vùng ghi được...")
        shutil.copytree(src, dst)
    return dst

TEST_DIR  = copy_pool_dir("pan-pc11-test-v1")
TRAIN_DIR = copy_pool_dir("pan-pc11-train-v1")
print("TEST_DIR =", TEST_DIR)
print("TRAIN_DIR =", TRAIN_DIR)

def approve_governance(dataset_dir: str, approved_at: str):
    manifest_path = pathlib.Path(dataset_dir) / "dataset-manifest.json"
    manifest = json.loads(manifest_path.read_text())
    manifest["governance"].update({
        "status": "APPROVED",
        "contains_pii": False,
        "retention": "Ephemeral Kaggle run; outputs giu lai lam bang chung thuc nghiem cho KLTN",
        "approver": "Vo Thi Kim Yen (tac gia KLTN, tu duyet du lieu cong khai PAN-PC-11 theo dieu khoan Zenodo)",
        "approved_at": approved_at,
    })
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
    print(dataset_dir, "-> governance:", manifest["governance"]["status"])

approve_governance(TEST_DIR, "2026-11-19")
approve_governance(TRAIN_DIR, "2026-11-20")

Đang copy pan-pc11-test-v1 sang vùng ghi được...
Đang copy pan-pc11-train-v1 sang vùng ghi được...
TEST_DIR = /kaggle/working/data/pan-pc11-test-v1
TRAIN_DIR = /kaggle/working/data/pan-pc11-train-v1
/kaggle/working/data/pan-pc11-test-v1 -> governance: APPROVED
/kaggle/working/data/pan-pc11-train-v1 -> governance: APPROVED


### Bước 4 — Chạy BM25 trên Test rồi Train

Mỗi lần chạy gọi `python -m research.offline_retrieval run` với config có sẵn `pan-pc-11-bm25-chunking-pilot-v1.json`, chạy cả 2 phương pháp (`fixed_window_bm25`, `structure_bm25`) trên cùng pool. Lấy top 100 tài liệu cho mỗi câu hỏi.

**Thời gian thực tế với bản BM25 nhanh:** khoảng **18 phút cho cả Test lẫn Train**.

Kết quả ghi vào `/kaggle/working/results/{test,train}/bm25_chunking/`, gồm:
- `predictions.jsonl`: danh sách tài liệu BM25 trả về cho từng câu hỏi (Train sẽ dùng làm nguồn khai thác cặp âm khó sau này);
- `metrics.json`: điểm Recall/nDCG/MRR;
- `run_manifest.json`: thông tin cấu hình lần chạy.

Kết quả đúng: `Test rc: 0 | Train rc: 0` (rc = 0 nghĩa là chạy không lỗi).

In [4]:
OUT_ROOT = "/kaggle/working/results"

with open(f"{REPO}/research/configs/retrieval/pan-pc-11-bm25-chunking-pilot-v1.json") as f:
    bm25_config = json.load(f)

def run_bm25(dataset_dir: str, split_name: str):
    cfg = dict(bm25_config)
    cfg["experiment_id"] = f"pan-pc-11-bm25-chunking-{split_name}-v1"
    cfg_path = f"/kaggle/working/{split_name}_bm25_config.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    out_dir = f"{OUT_ROOT}/{split_name}/bm25_chunking"
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "research.offline_retrieval", "run",
        "--dataset-manifest", f"{dataset_dir}/dataset-manifest.json",
        "--config", cfg_path,
        "--output-dir", out_dir,
    ]
    print("Đang chạy:", split_name, "...")
    result = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print("LỖI:", result.stderr[-3000:])
    return result.returncode

rc_test = run_bm25(TEST_DIR, "test")
rc_train = run_bm25(TRAIN_DIR, "train")
print("Test rc:", rc_test, "| Train rc:", rc_train)

Đang chạy: test ...
/kaggle/working/results/test/bm25_chunking/run_manifest.json

Đang chạy: train ...
/kaggle/working/results/train/bm25_chunking/run_manifest.json

Test rc: 0 | Train rc: 0


### Bước 5a — Xem nhanh cấu trúc `metrics.json`

Ô này chỉ in phần đầu file để kiểm tra kết quả đã được ghi ra. Trong mỗi phương pháp có:
- `macro`: điểm trung bình trên **từng câu hỏi**;
- `parent_macro`: điểm trung bình trên **từng parent**. Mỗi parent có thể có nhiều câu hỏi, nên gộp theo parent mới công bằng khi so sánh các nhóm;
- `per_query`: điểm của từng câu hỏi riêng lẻ (dùng cho bước tách nhóm và bootstrap).

In [5]:
import json
for split in ["test", "train"]:
    m = json.load(open(f"/kaggle/working/results/{split}/bm25_chunking/metrics.json"))
    print("=====", split, "=====")
    print(json.dumps(m, ensure_ascii=False, indent=1)[:3500])

===== test =====
{
 "fixed_window_bm25": {
  "macro": {
   "1": {
    "mrr": 0.7340752110514198,
    "ndcg": 0.7340752110514198,
    "recall": 0.7340752110514198
   },
   "10": {
    "mrr": 0.7511856034304231,
    "ndcg": 0.7600545463066468,
    "recall": 0.7881811204911742
   },
   "100": {
    "mrr": 0.7535579731486597,
    "ndcg": 0.7727383280480646,
    "recall": 0.8503453568687643
   },
   "20": {
    "mrr": 0.752532541846898,
    "ndcg": 0.7649880071850087,
    "recall": 0.8077513430544896
   },
   "5": {
    "mrr": 0.7495587106676899,
    "ndcg": 0.7560217547245546,
    "recall": 0.7755180353031466
   }
  },
  "parent_annotation_count": 300,
  "parent_macro": {
   "1": {
    "mrr": 0.8408953426990241,
    "ndcg": 0.8408953426990241,
    "recall": 0.8408953426990241
   },
   "10": {
    "mrr": 0.85020293336729,
    "ndcg": 0.8548944400943927,
    "recall": 0.869787732120606
   },
   "100": {
    "mrr": 0.8513803552096506,
    "ndcg": 0.8609412402884279,
    "recall": 0.8988630275

### Bước 5b — Bảng Recall tổng (chưa tách theo loại đạo văn)

**Recall@k:** với mỗi câu hỏi, trong top k tài liệu BM25 trả về có tìm thấy tài liệu nguồn đúng không (1 = có, 0 = không); lấy trung bình trên tất cả. R@1 chặt nhất (phải đúng ngay hạng 1); R@100 rộng nhất.

**Kết quả:**

| Pool | Phương pháp | R@1 | R@100 (mức parent) |
|---|---|---:|---:|
| Test | fixed_window | 0,841 | 0,899 |
| Test | structure | 0,819 | 0,881 |
| Train | fixed_window | 0,839 | 0,891 |
| Train | structure | 0,814 | 0,874 |

**Cách đọc:** cắt cửa sổ cố định nhỉnh hơn cắt theo cấu trúc ở cả hai pool. Cột `stage_timings: None` là bình thường vì runner không ghi thời gian từng bước.

In [7]:
import json
for split in ["test", "train"]:
    d = f"/kaggle/working/results/{split}/bm25_chunking"
    m = json.load(open(f"{d}/metrics.json"))
    print("=====", split, "=====")
    for method, v in m.items():
        if not isinstance(v, dict) or "macro" not in v:
            continue
        for scope in ["macro", "parent_macro"]:
            row = {k: round(v[scope][k]["recall"], 4) for k in ["1", "5", "10", "20", "100"]}
            print(f"{method:20s} {scope:13s} recall@k = {row}")
    rm = json.load(open(f"{d}/run_manifest.json"))
    print("stage_timings:", rm.get("stage_timings") or rm.get("timings"))

===== test =====
fixed_window_bm25    macro         recall@k = {'1': 0.7341, '5': 0.7755, '10': 0.7882, '20': 0.8078, '100': 0.8503}
fixed_window_bm25    parent_macro  recall@k = {'1': 0.8409, '5': 0.8625, '10': 0.8698, '20': 0.8801, '100': 0.8989}
structure_bm25       macro         recall@k = {'1': 0.6892, '5': 0.7206, '10': 0.7387, '20': 0.7536, '100': 0.8135}
structure_bm25       parent_macro  recall@k = {'1': 0.8187, '5': 0.8371, '10': 0.8458, '20': 0.8535, '100': 0.8811}
stage_timings: None
===== train =====
fixed_window_bm25    macro         recall@k = {'1': 0.7321, '5': 0.7631, '10': 0.7723, '20': 0.7834, '100': 0.8168}
fixed_window_bm25    parent_macro  recall@k = {'1': 0.839, '5': 0.8604, '10': 0.8648, '20': 0.8718, '100': 0.891}
structure_bm25       macro         recall@k = {'1': 0.6867, '5': 0.724, '10': 0.7344, '20': 0.7499, '100': 0.7844}
structure_bm25       parent_macro  recall@k = {'1': 0.8135, '5': 0.8348, '10': 0.8426, '20': 0.8523, '100': 0.8736}
stage_timings: None


### Bước 6a — Lấy nhãn loại obfuscation cho từng parent

PAN-PC-11 gắn cho mỗi đoạn đạo văn 2 nhãn trong file XML gốc:
- `type`: cách tạo đạo văn: **artificial** (máy tự đổi từ/đảo câu), **translation** (dịch từ tiếng Đức/Tây Ban Nha sang tiếng Anh), **simulated** (người viết lại thật);
- `obfuscation`: mức xáo trộn (`none` / `low` / `high`). Nhóm `None` (không có nhãn mức) thực chất là translation + simulated, vì XML không ghi mức cho 2 loại này.

Ô này lần theo `annotation_provenance` của từng câu hỏi để tìm lại đúng thẻ trong XML, rồi in số lượng mỗi nhóm.

Kết quả: Test có 256 artificial / 23 translation / 21 simulated; Train có 430 / 51 / 19. Hai nhóm nhỏ (translation, simulated) khiến kết luận về chúng kém chắc hơn.

In [8]:
import glob, json, xml.etree.ElementTree as ET

ext = glob.glob('/kaggle/input/**/external-detection-corpus', recursive=True)
assert ext, "Chưa gắn dataset có external-detection-corpus vào notebook này"
INPUT_ROOT = ext[0]

def build_labels(pool_dir):
    parent_to_prov = {}
    for line in open(f"{pool_dir}/queries.jsonl", encoding="utf-8"):
        q = json.loads(line)
        parent_to_prov.setdefault(q["parent_annotation_id"], q["annotation_provenance"])
    labels = {}
    for pid, prov in parent_to_prov.items():
        root = ET.parse(f"{INPUT_ROOT}/{prov['xml_path']}").getroot()
        feats = [f for f in root.iter()
                 if f.tag.split('}')[-1] == 'feature' and f.attrib.get('name', '').casefold() == 'plagiarism']
        feat = feats[prov["feature_index"]]
        labels[pid] = {"type": feat.attrib.get("type"), "obfuscation": feat.attrib.get("obfuscation")}
    return labels

LABELS = {
    "test":  build_labels("/kaggle/working/data/pan-pc11-test-v1"),
    "train": build_labels("/kaggle/working/data/pan-pc11-train-v1"),
}
from collections import Counter
for s, lab in LABELS.items():
    print(s, "type:", Counter(v["type"] for v in lab.values()),
             "| obfuscation:", Counter(v["obfuscation"] for v in lab.values()))

test type: Counter({'artificial': 256, 'translation': 23, 'simulated': 21}) | obfuscation: Counter({'low': 131, 'high': 119, None: 44, 'none': 6})
train type: Counter({'artificial': 430, 'translation': 51, 'simulated': 19}) | obfuscation: Counter({'high': 209, 'low': 205, None: 70, 'none': 16})


### Bước 6b — Tách kết quả theo loại obfuscation, kèm khoảng tin cậy

Với mỗi loại, tính Recall trung bình **theo parent** (không phải theo câu hỏi), rồi dùng **bootstrap** để ước lượng độ chắc chắn: bốc ngẫu nhiên có hoàn lại các parent 2.000 lần, tính lại trung bình mỗi lần, lấy khoảng 95% ở giữa. Đây là **khoảng tin cậy 95%** (CI): khoảng càng hẹp thì số càng đáng tin.

Cách đọc: `n` là số parent trong nhóm; nhóm nhỏ thì CI rộng.

**Phát hiện chính:**
- **artificial:** R@100 ≈ 0,95, BM25 làm tốt.
- **translation:** R@100 chỉ ≈ 0,23 (Test) và 0,33 (Train). BM25 so khớp từ, mà bản dịch là tiếng Anh còn nguồn là tiếng Đức/Tây Ban Nha, nên gần như không khớp được. Đây là chỗ Dense đa ngôn ngữ và reranker đã fine-tune có thể chứng minh giá trị.
- **simulated:** luôn bằng 1,0, không còn chỗ để phân biệt phương pháp. CI `[1,0, 1,0]` chỉ là hệ quả toán học khi mọi parent cùng một giá trị, đừng đọc là "chắc chắn tuyệt đối".

In [9]:
import numpy as np
from collections import defaultdict

def breakdown(split, key="type", ks=("1", "10", "100"), n_boot=2000, seed=20261121):
    pool = f"/kaggle/working/data/pan-pc11-{split}-v1"
    qid2pid = {}
    for line in open(f"{pool}/queries.jsonl", encoding="utf-8"):
        q = json.loads(line); qid2pid[q["query_id"]] = q["parent_annotation_id"]
    metrics = json.load(open(f"/kaggle/working/results/{split}/bm25_chunking/metrics.json"))
    rng = np.random.default_rng(seed)
    for method, data in metrics.items():
        if not isinstance(data, dict) or "per_query" not in data:
            continue
        print(f"\n===== {split} | {method} | nhóm theo '{key}' =====")
        for k in ks:
            per_parent = defaultdict(list)
            for qid, pid in qid2pid.items():
                if qid in data["per_query"] and k in data["per_query"][qid]:
                    per_parent[pid].append(data["per_query"][qid][k]["recall"])
            groups = defaultdict(list)
            for pid, rec in per_parent.items():
                groups[LABELS[split][pid][key]].append(np.mean(rec))
            groups["TẤT CẢ"] = [v for g in list(groups.values()) for v in g]
            for name, vals in sorted(groups.items(), key=lambda x: -len(x[1])):
                vals = np.array(vals)
                boots = rng.choice(vals, size=(n_boot, len(vals)), replace=True).mean(axis=1)
                lo, hi = np.percentile(boots, [2.5, 97.5])
                print(f"  R@{k:>3} | {str(name):12s} n={len(vals):4d}  mean={vals.mean():.4f}  95%CI=[{lo:.4f}, {hi:.4f}]")

for split in ["test", "train"]:
    breakdown(split, key="type")
breakdown("test", key="obfuscation")   # phụ: theo mức obfuscation none/low/high


===== test | fixed_window_bm25 | nhóm theo 'type' =====
  R@  1 | TẤT CẢ       n= 300  mean=0.8409  95%CI=[0.8027, 0.8762]
  R@  1 | artificial   n= 256  mean=0.8880  95%CI=[0.8546, 0.9186]
  R@  1 | translation  n=  23  mean=0.1712  95%CI=[0.0571, 0.3143]
  R@  1 | simulated    n=  21  mean=1.0000  95%CI=[1.0000, 1.0000]
  R@ 10 | TẤT CẢ       n= 300  mean=0.8698  95%CI=[0.8362, 0.9028]
  R@ 10 | artificial   n= 256  mean=0.9185  95%CI=[0.8899, 0.9457]
  R@ 10 | translation  n=  23  mean=0.2082  95%CI=[0.0776, 0.3638]
  R@ 10 | simulated    n=  21  mean=1.0000  95%CI=[1.0000, 1.0000]
  R@100 | TẤT CẢ       n= 300  mean=0.8989  95%CI=[0.8687, 0.9279]
  R@100 | artificial   n= 256  mean=0.9508  95%CI=[0.9279, 0.9713]
  R@100 | translation  n=  23  mean=0.2288  95%CI=[0.0991, 0.3792]
  R@100 | simulated    n=  21  mean=1.0000  95%CI=[1.0000, 1.0000]

===== test | structure_bm25 | nhóm theo 'type' =====
  R@  1 | TẤT CẢ       n= 300  mean=0.8187  95%CI=[0.7786, 0.8551]
  R@  1 | artifici

### Bước 7 — Bootstrap cặp: fixed-window có thật sự hơn structure không?

Bước 6 so khoảng tin cậy của từng phương pháp riêng, nhưng 2 khoảng chồng nhau không có nghĩa hai phương pháp không khác nhau (vì chúng chạy trên cùng các parent). Cách đúng là **tính hiệu số trên từng parent** (fixed − structure), rồi bootstrap trên hiệu số đó.

**Cách đọc cột cuối "loại 0?":** nếu khoảng tin cậy của hiệu số **không chứa 0**, ta nói chênh lệch có ý nghĩa thống kê (không phải ngẫu nhiên).

**Kết quả:**
- Tổng thể và nhóm artificial: fixed hơn structure có ý nghĩa thống kê ở cả Test và Train, cả 3 mức k. Mức hơn nhỏ, khoảng **+1,4 đến +2,5 điểm phần trăm**.
- Translation: cùng chiều và CI cũng không chứa 0, nhưng n chỉ 23 (Test) và 51 (Train). Ở R@1 Test cận dưới chỉ +0,0016, sát 0, nên chỉ nói "xu hướng cùng chiều".
- Simulated: hiệu số bằng 0 (cả hai đều 1,0), không có thông tin so sánh.
- Chưa hiệu chỉnh cho việc so nhiều nhóm và nhiều k cùng lúc, nên phần tách nhóm nhỏ mang tính thăm dò.

**Đầu ra:** ghi `paired_bootstrap.json` và `paired_bootstrap.md` vào từng thư mục `bm25_chunking`, rồi nén toàn bộ `results` thành `bm25_chunking_results.zip`. **Đây là file cần tải về.**

In [12]:
# Bootstrap cặp: fixed_window_bm25 − structure_bm25, mức parent, theo loại obfuscation
import json, zipfile, os
import numpy as np

RES = "/kaggle/working/results"
KS = ("1", "10", "100")
N_BOOT, SEED = 2000, 20261121
POOLS = {"test": TEST_DIR, "train": TRAIN_DIR}   # biến đã có từ ô trước

def find_parent(obj, labels, depth=0):
    """Tìm giá trị chuỗi trong query khớp một mã parent có nhãn."""
    if isinstance(obj, str):
        return obj if obj in labels else None
    if depth > 3:
        return None
    items = obj.values() if isinstance(obj, dict) else obj if isinstance(obj, list) else []
    for v in items:
        r = find_parent(v, labels, depth + 1)
        if r:
            return r
    return None

def load_queries(pool_dir):
    qs = []
    with open(f"{pool_dir}/queries.jsonl", encoding="utf-8") as f:
        for line in f.read().split("\n"):
            if line.strip():
                qs.append(json.loads(line))
    return qs

def parent_means(per_query, q2p, parents):
    """parent -> {k: recall trung bình trên các câu hỏi của parent}"""
    acc = {p: {k: [] for k in KS} for p in parents}
    for qid, v in per_query.items():
        p = q2p.get(qid)
        if p is None:
            continue
        for k in KS:
            acc[p][k].append(v[k]["recall"])
    return {p: {k: float(np.mean(acc[p][k])) for k in KS} for p in parents if acc[p]["1"]}

def boot_diff(d, rng):
    d = np.asarray(d)
    idx = rng.integers(0, len(d), size=(N_BOOT, len(d)))
    means = d[idx].mean(axis=1)
    return float(d.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

report = {}
for split, pool_dir in POOLS.items():
    labels = build_labels(pool_dir)
    queries = load_queries(pool_dir)
    q2p = {}
    for q in queries:
        p = find_parent(q, labels)
        if p:
            q2p[q["query_id"]] = p
    print(f"[{split}] ghép được {len(q2p)}/{len(queries)} câu hỏi vào parent")
    if len(q2p) < len(queries):
        print("  CẢNH BÁO: có câu hỏi không ghép được; ví dụ một query:",
              json.dumps(queries[0], ensure_ascii=False)[:500])
        raise SystemExit("Dừng: cần xem lại trường chứa mã parent.")

    m = json.load(open(f"{RES}/{split}/bm25_chunking/metrics.json", encoding="utf-8"))
    parents = sorted(set(q2p.values()))
    fx = parent_means(m["fixed_window_bm25"]["per_query"], q2p, parents)
    st = parent_means(m["structure_bm25"]["per_query"], q2p, parents)
    common = [p for p in parents if p in fx and p in st]

    groups = {"all": common}
    for t in ("artificial", "translation", "simulated"):
        groups[t] = [p for p in common if labels[p]["type"] == t]

    rng = np.random.default_rng(SEED)
    out = {}
    for g, ps in groups.items():
        out[g] = {"n_parent": len(ps)}
        for k in KS:
            d = [fx[p][k] - st[p][k] for p in ps]
            if not d:
                continue
            mean, lo, hi = boot_diff(d, rng)
            out[g][f"R@{k}"] = {
                "fixed": float(np.mean([fx[p][k] for p in ps])),
                "structure": float(np.mean([st[p][k] for p in ps])),
                "diff": mean, "ci95": [lo, hi],
                "excludes_zero": bool(lo > 0 or hi < 0),
            }
    report[split] = out

    d_out = f"{RES}/{split}/bm25_chunking"
    json.dump(out, open(f"{d_out}/paired_bootstrap.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)
    lines = [f"# Bootstrap cặp fixed − structure ({split}), mức parent",
             f"seed {SEED}, {N_BOOT} lần, CI 95% theo parent", "",
             "| nhóm | n | chỉ số | fixed | structure | hiệu | CI 95% | loại 0? |",
             "|---|---:|---|---:|---:|---:|---|---|"]
    for g, v in out.items():
        for k in KS:
            r = v.get(f"R@{k}")
            if r:
                lines.append(f"| {g} | {v['n_parent']} | R@{k} | {r['fixed']:.4f} | {r['structure']:.4f} | "
                             f"{r['diff']:+.4f} | [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}] | "
                             f"{'có' if r['excludes_zero'] else 'không'} |")
    open(f"{d_out}/paired_bootstrap.md", "w", encoding="utf-8").write("\n".join(lines))
    print("\n".join(lines), "\n")

# Nén lại toàn bộ results thành một file zip
ZIP = "/kaggle/working/bm25_chunking_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(RES):
        for fn in files:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, "/kaggle/working"))
print("Đã ghi:", ZIP, f"({os.path.getsize(ZIP)/1e6:.1f} MB)")

[test] ghép được 2606/2606 câu hỏi vào parent
# Bootstrap cặp fixed − structure (test), mức parent
seed 20261121, 2000 lần, CI 95% theo parent

| nhóm | n | chỉ số | fixed | structure | hiệu | CI 95% | loại 0? |
|---|---:|---|---:|---:|---:|---|---|
| all | 300 | R@1 | 0.8409 | 0.8187 | +0.0222 | [+0.0140, +0.0304] | có |
| all | 300 | R@10 | 0.8698 | 0.8458 | +0.0240 | [+0.0161, +0.0322] | có |
| all | 300 | R@100 | 0.8989 | 0.8811 | +0.0177 | [+0.0104, +0.0266] | có |
| artificial | 256 | R@1 | 0.8880 | 0.8633 | +0.0247 | [+0.0151, +0.0364] | có |
| artificial | 256 | R@10 | 0.9185 | 0.8937 | +0.0249 | [+0.0165, +0.0343] | có |
| artificial | 256 | R@100 | 0.9508 | 0.9326 | +0.0181 | [+0.0097, +0.0280] | có |
| translation | 23 | R@1 | 0.1712 | 0.1557 | +0.0155 | [+0.0016, +0.0325] | có |
| translation | 23 | R@10 | 0.2082 | 0.1715 | +0.0366 | [+0.0109, +0.0651] | có |
| translation | 23 | R@100 | 0.2288 | 0.1994 | +0.0294 | [+0.0070, +0.0551] | có |
| simulated | 21 | R@1 | 1.0000 |